In [4]:
import numpy as np
from scipy.stats import wilcoxon

# --- Dados da Instância (Exercício 2 da Aula 1) ---
# Substitua pelos arrays exatos do seu PDF caso difiram

N_ITENS = 20
np.random.seed(42)
pesos = np.random.randint(1, 16, size=N_ITENS)
valores = np.random.randint(1, 21, size=N_ITENS)
CAPACIDADE = np.sum(pesos)/2

# --- Função de Avaliação (Fitness) ---
def calcular_fitness(individuo):
    peso_total = np.sum(individuo * pesos)
    if peso_total > CAPACIDADE:
        return 0 # Penalização
    return np.sum(individuo * valores)

# ==========================================
# 1. HEURÍSTICA GULOSA (Razão Valor/Peso)
# ==========================================
def executar_gulosa():
    razao = valores / pesos
    indices_ordenados = np.argsort(razao)[::-1] # Do maior para o menor

    mochila = np.zeros(N_ITENS)
    peso_atual = 0

    for idx in indices_ordenados:
        if peso_atual + pesos[idx] <= CAPACIDADE:
            mochila[idx] = 1
            peso_atual += pesos[idx]

    return calcular_fitness(mochila)

# ==========================================
# 2. BUSCA ALEATÓRIA (10.000 NFE por rodada)
# ==========================================
def executar_busca_aleatoria(nfe=10000):
    melhor_fitness = 0
    for _ in range(nfe):
        ind = np.random.randint(2, size=N_ITENS)
        fit = calcular_fitness(ind)
        if fit > melhor_fitness:
            melhor_fitness = fit
    return melhor_fitness

# ==========================================
# 3. ALGORITMO GENÉTICO (Exercício 1 - Aula 5)
# ==========================================
def executar_ag(n_pop=50, n_gen=200): # 50 * 200 = 10.000 NFE
    pop = np.random.randint(2, size=(n_pop, N_ITENS))
    p_c, p_m = 0.8, 1.0 / N_ITENS

    for _ in range(n_gen):
        fits = np.array([calcular_fitness(ind) for ind in pop])
        nova_pop = [pop[np.argmax(fits)].copy()] # Elitismo

        while len(nova_pop) < n_pop:
            # Torneio
            t1, t2 = np.random.choice(n_pop, 2, replace=False)
            p1 = pop[t1] if fits[t1] > fits[t2] else pop[t2]

            t3, t4 = np.random.choice(n_pop, 2, replace=False)
            p2 = pop[t3] if fits[t3] > fits[t4] else pop[t4]

            # Crossover 1 ponto
            if np.random.rand() < p_c:
                pt = np.random.randint(1, N_ITENS)
                f1, f2 = np.concatenate([p1[:pt], p2[pt:]]), np.concatenate([p2[:pt], p1[pt:]])
            else:
                f1, f2 = p1.copy(), p2.copy()

            # Mutação Bit-flip
            for i in range(N_ITENS):
                if np.random.rand() < p_m: f1[i] = 1 - f1[i]
                if np.random.rand() < p_m: f2[i] = 1 - f2[i]

            nova_pop.append(f1)
            if len(nova_pop) < n_pop: nova_pop.append(f2)

        pop = np.array(nova_pop)

    fits_finais = np.array([calcular_fitness(ind) for ind in pop])
    return np.max(fits_finais)

# --- Execução do Experimento e Teste Estatístico ---
print("Executando experimentos (30 rodadas)...")
resultado_gulosa = executar_gulosa() # Determinístico, roda 1 vez
resultados_aleatoria = [executar_busca_aleatoria() for _ in range(30)]
resultados_ag = [executar_ag() for _ in range(30)]

# Análise de Wilcoxon (AG vs Aleatória)
stat, p_valor = wilcoxon(resultados_ag, resultados_aleatoria)

print(f"\n--- RESULTADOS ---")
print(f"Gulosa (Fixo): {resultado_gulosa}")
print(f"Aleatória (Média): {np.mean(resultados_aleatoria):.2f}")
print(f"Aleatória (Melhor): {max(resultados_aleatoria):.2f}")
print(f"AG (Média): {np.mean(resultados_ag):.2f}")
print(f"AG (Melhor): {max(resultados_ag):.2f}")
print(f"\n--- TESTE DE WILCOXON (AG vs Aleatória) ---")
print(f"Estatística: {stat}")
print(f"P-valor: {p_valor:.5e}")

if p_valor < 0.05:
    print("Conclusão: O AG é ESTATISTICAMENTE SUPERIOR à Busca Aleatória (p < 0.05).")
else:
    print("Conclusão: Não há diferença estatística significativa.")

Executando experimentos (30 rodadas)...

--- RESULTADOS ---
Gulosa (Fixo): 169.0
Aleatória (Média): 158.57
Aleatória (Melhor): 166.00
AG (Média): 169.00
AG (Melhor): 169.00

--- TESTE DE WILCOXON (AG vs Aleatória) ---
Estatística: 0.0
P-valor: 1.65100e-06
Conclusão: O AG é ESTATISTICAMENTE SUPERIOR à Busca Aleatória (p < 0.05).
